In [1]:
pip install torch transformers datasets scikit-learn pandas numpy sentencepiece

In [ ]:
import os
import random
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.metrics import accuracy_score, f1_score
from sklearn.preprocessing import LabelEncoder
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)
train_df = pd.read_csv("final_train.csv")
val_df = pd.read_csv("final_validation.csv")
test_df = pd.read_csv("test_input.csv")


label_encoder = LabelEncoder()
train_df["encoded_label"] = label_encoder.fit_transform(train_df["label"])
val_df["encoded_label"] = label_encoder.transform(val_df["label"])
num_labels = len(label_encoder.classes_)

train_dataset = Dataset.from_pandas(
    train_df[["sentence", "encoded_label"]].rename(columns={"encoded_label": "label"})
)
val_dataset = Dataset.from_pandas(
    val_df[["sentence", "encoded_label"]].rename(columns={"encoded_label": "label"})
)
test_dataset = Dataset.from_pandas(test_df[["sentence"]])

model_name = "answerdotai/ModernBERT-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

def tokenize_function(examples):
    return tokenizer(examples["sentence"], truncation=True, max_length=160)

tokenized_train = train_dataset.map(tokenize_function, batched=True)
tokenized_val = val_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

keep_cols_train = ["input_ids", "attention_mask", "label"]
keep_cols_test = ["input_ids", "attention_mask"]
if "token_type_ids" in tokenized_train.column_names:
    keep_cols_train.append("token_type_ids")
    keep_cols_test.append("token_type_ids")

tokenized_train = tokenized_train.remove_columns([c for c in tokenized_train.column_names if c not in keep_cols_train])
tokenized_val = tokenized_val.remove_columns([c for c in tokenized_val.column_names if c not in keep_cols_train])
tokenized_test = tokenized_test.remove_columns([c for c in tokenized_test.column_names if c not in keep_cols_test])


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)
    return {
        "accuracy": accuracy_score(labels, predictions),
        "f1_macro": f1_score(labels, predictions, average="macro"),
        "f1_weighted": f1_score(labels, predictions, average="weighted"),
    }


use_fp16 = torch.cuda.is_available()

training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    gradient_accumulation_steps=2,
    num_train_epochs=6,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    logging_strategy="steps",
    logging_steps=25,
    report_to="none",
    fp16=False,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()

val_metrics = trainer.evaluate(tokenized_val)
for k, v in val_metrics.items():
    if isinstance(v, (float, np.floating)):
        print(f"   {k}: {v:.4f}")
    else:
        print(f"   {k}: {v}")


pred_output = trainer.predict(tokenized_test)
pred_classes = np.argmax(pred_output.predictions, axis=1)
pred_labels = label_encoder.inverse_transform(pred_classes)

submission_df = test_df.copy()
submission_df["label"] = pred_labels
submission_df = submission_df[["sentence", "label"]]
submission_df.to_csv("predictions.csv", index=False)


1. Loading data...
   Train rows:      12104
   Validation rows: 4035
   Test rows:       4035
2. Loading tokenizer and model (modernbert-base)...


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


3. Tokenizing datasets...


Map:   0%|          | 0/12104 [00:00<?, ? examples/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Epoch,Training Loss,Validation Loss,Accuracy,F1 Macro,F1 Weighted
1,0.741255,0.335970,0.904089,0.701815,0.896398
2,0.282527,0.148516,0.961090,0.913186,0.960432
3,0.044969,0.135797,0.967534,0.957146,0.967020
4,0.060935,0.167315,0.971747,0.940655,0.970513
5,0.031312,0.157894,0.975217,0.950091,0.974448


6. Evaluating best checkpoint on validation split...


Validation metrics:
   eval_loss: 0.1358
   eval_accuracy: 0.9675
   eval_f1_macro: 0.9571
   eval_f1_weighted: 0.9670
   eval_runtime: 10.2322
   eval_samples_per_second: 394.3440
   eval_steps_per_second: 12.4120
   epoch: 5.0000
7. Generating test predictions...
Saved predictions to: predictions.csv


In [ ]:

import os
import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.preprocessing import LabelEncoder
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
)


train_df = pd.read_csv("final_train.csv")
val_df = pd.read_csv("final_validation.csv")
test_df = pd.read_csv("test_input.csv")

combined_df = pd.concat([train_df, val_df], ignore_index=True)


label_encoder = LabelEncoder()
combined_df["encoded_label"] = label_encoder.fit_transform(combined_df["label"])
num_labels = len(label_encoder.classes_)

combined_dataset = Dataset.from_pandas(
    combined_df[["sentence", "encoded_label"]].rename(columns={"encoded_label": "label"})
)
test_dataset = Dataset.from_pandas(test_df[["sentence"]])


model_name = "answerdotai/ModernBERT-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
final_model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=num_labels)

def tokenize_function(examples):
    return tokenizer(examples["sentence"], truncation=True, max_length=160)


tokenized_combined = combined_dataset.map(tokenize_function, batched=True)
tokenized_test = test_dataset.map(tokenize_function, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

keep_cols_train = ["input_ids", "attention_mask", "label"]
keep_cols_test = ["input_ids", "attention_mask"]
if "token_type_ids" in tokenized_combined.column_names:
    keep_cols_train.append("token_type_ids")
    keep_cols_test.append("token_type_ids")

tokenized_combined = tokenized_combined.remove_columns([c for c in tokenized_combined.column_names if c not in keep_cols_train])
tokenized_test = tokenized_test.remove_columns([c for c in tokenized_test.column_names if c not in keep_cols_test])


final_training_args = TrainingArguments(
    output_dir="./final_results",
    learning_rate=5e-5,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=2,
    num_train_epochs=8,              
    warmup_ratio=0.1,
    weight_decay=0.01,
    save_strategy="no",              
    logging_strategy="steps",
    logging_steps=25,
    report_to="none",
    fp16=False,
)

final_trainer = Trainer(
    model=final_model,
    args=final_training_args,
    train_dataset=tokenized_combined,
    data_collator=data_collator,
)


final_trainer.train()

pred_output = final_trainer.predict(tokenized_test)
pred_classes = np.argmax(pred_output.predictions, axis=1)
pred_labels = label_encoder.inverse_transform(pred_classes)

submission_df = test_df.copy()
submission_df["label"] = pred_labels
submission_df = submission_df[["sentence", "label"]]
submission_df.to_csv("predictions_final.csv", index=False)


1. Loading and combining data...
   Combined Train rows: 16139
   Test rows:           4035
2. Loading tokenizer and fresh model (ModernBERT-base)...


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.weight | MISSING    | 
classifier.bias   | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


3. Tokenizing datasets...


Map:   0%|          | 0/16139 [00:00<?, ? examples/s]

...

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


4. Setting up final training arguments...
5. Fine-tuning ModernBERT on combined dataset...


Step,Training Loss
25,4.508527
50,2.715098
75,2.070098
100,1.816463
125,1.770038
150,1.682020
175,1.415135
200,1.405308
225,1.341785
250,1.276095


6. Generating final test predictions...


Saved predictions to: predictions_final.csv
